# From Time-Series to Frequency-Domain


## Environment set up

Change the working directory to be able to work with the source-code of this repository.

In [1]:
import os
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().parents[0]
os.chdir(WORKING_DIRECTORY)

## Imports

In [2]:
from src.read import read_nasa_vibration_files_in_directory
from src.signals.processing import Signal, band_pass_filter, low_pass_filter, high_pass_filter, envelope, power_spectrum, process_signal
from src.signals import calculations
import matplotlib.pyplot as plt
import numpy as np
from loguru import logger
import matplotlib.dates as mdates
import polars as pl
import plotly.express as px
from datetime import datetime

## Inputs

The inputs have been obtained from the NASA bearings documentation.

The following cell displays the data path for each test and the name of their columns:

In [3]:
DATA_INPUTS_PER_TEST = {
    '1st_test': {'data_path': 'data/nasa_ims_bearing_dataset/1st_test',
                  'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4',
                                   'channel_5', 'channel_6', 'channel_7', 'channel_8']},
    '2nd_test': {'data_path': 'data/nasa_ims_bearing_dataset/2nd_test',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']},
    '3rd_test': {'data_path': 'data/nasa_ims_bearing_dataset/3rd_test/4th_test/txt',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']}
          }

As each test has a different set up of sensors or channels per bearing, the following cell describes them:

In [4]:
BEARING_CHANNEL_MAPPING = {
    '1st_test': {'bearing_1': ['channel_1', 'channel_2'],
                 'bearing_2': ['channel_3', 'channel_4'],
                 'bearing_3': ['channel_5', 'channel_6'],
                 'bearing_4': ['channel_7', 'channel_8']},
    '2nd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']},
    '3rd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']}                 
}

Next, the faulty bearings are defined per test:

In [5]:
FAULTY_BEARINGS_PER_TEST = {
    '1st_test': {'bearing_3': 'bearing_inner_race',
                 'bearing_4': 'bearing_roller'
                 },
    '2nd_test': {'bearing_1': 'bearing_outer_race'},
    '3rd_test': {'bearing_3': 'bearing_outer_race'}
}

As final inputs, the following parameters are needed to read properly the vibration signals. In addition, an acceptable sensor range is defined to avoid faulty channel signals:

In [6]:
SAMPLING_FREQUENCY = 20000
MEASUREMENT_DURATION_IN_SECONDS = 1
ACCEPTABLE_SENSOR_RANGE = 0.01

## Read the data

In [7]:
complete_data_path_per_test = {}

for test, inputs_per_test in DATA_INPUTS_PER_TEST.items():
    for key, values in inputs_per_test.items():
        data_path = inputs_per_test['data_path']
        complete_path = WORKING_DIRECTORY.joinpath(data_path)
        complete_data_path_per_test[test] = complete_path


In [8]:
signal_resolution = calculations.resolution(sampling_frequency=SAMPLING_FREQUENCY)

df_list_per_test = {}
for test, file_path in complete_data_path_per_test.items():
    logger.info(f'test: {test}')
    column_names = DATA_INPUTS_PER_TEST[test]['column_names']
    df_list = read_nasa_vibration_files_in_directory(files_path=file_path, sensors=column_names,
                                                     signal_resolution=signal_resolution,
                                                     acceptable_sensor_range=ACCEPTABLE_SENSOR_RANGE)
    df_list_sorted = sorted(df_list, key=lambda df: datetime.strptime(df['file_name'][0], '%Y.%m.%d.%H.%M.%S')) 
    df_list_per_test[test] = df_list_sorted

2026-02-05 20:10:40.254 | INFO     | __main__:<module>:5 - test: 1st_test
2026-02-05 20:10:42.480 | INFO     | src.read:read_nasa_vibration_files_in_directory:142 - 0 files were discarded.
2026-02-05 20:10:42.480 | INFO     | src.read:read_nasa_vibration_files_in_directory:143 - 1092 files were read successfully.
2026-02-05 20:10:42.580 | INFO     | __main__:<module>:5 - test: 2nd_test
2026-02-05 20:10:43.212 | WARNING  | src.read:read_nasa_vibration_files_in_directory:128 - All sensors in file 2004.02.19.06.22.39 are faulty for the defined acceptable_sensor_range of 0.01. Skipping this file.
2026-02-05 20:10:43.611 | WARNING  | src.read:read_nasa_vibration_files_in_directory:128 - All sensors in file 2004.02.19.06.12.39 are faulty for the defined acceptable_sensor_range of 0.01. Skipping this file.
2026-02-05 20:10:43.706 | INFO     | src.read:read_nasa_vibration_files_in_directory:142 - 2 files were discarded.
2026-02-05 20:10:43.707 | INFO     | src.read:read_nasa_vibration_files_in

## Signal Processing


In [9]:
SIGNAL_PROCESSING_STEPS = [  
    # {
    #     'type': 'low_pass_filter',
    #     'sampling_frequency': SAMPLING_FREQUENCY,
    #     'cutoff_frequency': 750
    # },    
    {
        'type': 'envelope',
        'remove_dc_offset': True
    },
    {
        'type': 'power_spectrum',
        'sampling_frequency': SAMPLING_FREQUENCY
    }
]

print(SIGNAL_PROCESSING_STEPS)

[{'type': 'envelope', 'remove_dc_offset': True}, {'type': 'power_spectrum', 'sampling_frequency': 20000}]


In [10]:
processed_signals = []
for test_df in df_list_per_test['2nd_test'][:4]:
    file_name = test_df['file_name'][0]
    y = test_df['channel_1'].to_numpy()
    x = test_df['measurement_time_in_seconds'].to_numpy()
    signal = Signal(x=x, y=y)
    processed_signal = process_signal(signal=signal, steps=SIGNAL_PROCESSING_STEPS)
    fig = px.line(x=processed_signal.x,
                  y=processed_signal.y,
                  markers=True,
                  title=f'Test Signal Scatter Plot - {file_name}',
                  labels={'x': 'Frequency (Hz)', 'y': 'Amplitude'},
                  width=1000,
                  height=500
    )
    fig.show();

In [11]:
processed_signals = []
for test_df in df_list_per_test['2nd_test'][-4:]:
    file_name = test_df['file_name'][0]
    y = test_df['channel_1'].to_numpy()
    x = test_df['measurement_time_in_seconds'].to_numpy()
    signal = Signal(x=x, y=y)
    processed_signal = process_signal(signal=signal, steps=SIGNAL_PROCESSING_STEPS)
    fig = px.line(x=processed_signal.x,
                  y=processed_signal.y,
                  markers=True,
                  title=f'Test Signal Scatter Plot - {file_name}',
                  labels={'x': 'Frequency (Hz)', 'y': 'Amplitude'},
                  width=1000,
                  height=500
    )
    fig.show();